In [6]:
import pandas as pd
import numpy as np
import math
from IPython.display import display, HTML

# ---------------------------------------------------------
# 1. ฟังก์ชันคำนวณ MRP (ฉบับเลียนแบบสูตรในสไลด์อาจารย์)
# ---------------------------------------------------------
def calculate_mrp_item_slide_logic(item_name, periods, gr, sr, initial_oh, lot_size, lead_time):
    n = len(periods)
    oh, nr, por_uncorrected, por, pei, prel = ([0]*n for _ in range(6))
    
    for i in range(n):
        # On Hand ต้นงวด
        oh[i] = initial_oh if i == 0 else pei[i-1]
        
        # Net Requirement
        nr_val = gr[i] - oh[i] - sr[i]
        nr[i] = nr_val
        
        # POR
        if nr_val > 0:
            por_uncorrected[i] = nr_val
            if lot_size == 'L4L':
                por[i] = nr_val
            else:
                multiplier = math.ceil(nr_val / lot_size)
                por[i] = int(multiplier * lot_size)
        else:
            por_uncorrected[i] = 0
            por[i] = 0
            
        # PEI ปลายงวด
        pei[i] = max(0, por[i] + sr[i] + oh[i] - gr[i])
        
    for i in range(n):
        prel[i] = por[i + lead_time] if i + lead_time < n else 0
            
    df = pd.DataFrame({
        'ช่วงเวลา': periods,
        'Gross Requirement (GR)': gr,
        'Scheduled Receipts (SR)': sr,
        'On Hand (OH)': oh,
        'Net Requirement (NR)': nr,
        'Planned Order Receipts uncorrected': por_uncorrected,
        'Planned Order Receipts (POR), corrected for lot size': por,
        'Planned End Inventory (PEI)': pei,
        'Planned Order Releases (PREL)': prel
    }).T
    
    df.columns = df.iloc[0].astype(int)
    return df[1:], nr # ส่งค่า NR (Net Requirement) กลับไปเพื่อคำนวณ GR ของลูก

# ---------------------------------------------------------
# 2. ฟังก์ชันรันระบบ MRP (ระเบิด BOM ตามสมการในสไลด์)
# ---------------------------------------------------------
def run_mrp_system_slide_logic(periods, mps, bom, item_master, process_order):
    gr_dict = {item: list(mps.get(item, [0]*len(periods))) for item in item_master}
    nr_dict = {}
    
    display(HTML("<h2>ผลการวิเคราะห์ความต้องการวัสดุ (ตรงกับเฉลยในสไลด์)</h2>"))
    
    for item in process_order:
        info = item_master[item]
        df, nr_out = calculate_mrp_item_slide_logic(
            item, periods, gr_dict[item], info['SR'], 
            info['OH'], info['LotSize'], info['LT']
        )
        nr_dict[item] = nr_out
        
        display(HTML(f"<br><b>รายการ: {item} | Lot Size: {info['LotSize']} | Lead Time: {info['LT']}</b>"))
        display(df)
        
        # **จุดที่ปรับแก้ให้ตรงกับสไลด์**
        # GR ลูก[i] = NR แม่[i + Lead Time ลูก] * จำนวนชิ้น
        if item in bom:
            for child, qty in bom[item].items():
                child_lt = item_master[child]['LT']
                for i in range(len(periods)):
                    parent_period = i + child_lt # ดึง NR แม่แบบข้าม Lead Time ลูกมาเป็นเป้าหมาย
                    if parent_period < len(periods): # กัน Index Out of Bounds
                        if nr_dict[item][parent_period] > 0:
                            gr_dict[child][i] += nr_dict[item][parent_period] * qty


# ฟังก์ชันคำนวณ CRP
def run_crp_system(days, por_units, operations, capacity_per_day):
    crp_data = {'Schedule (day)': days, 'POR (Unit)': por_units}
    total_time = [0] * len(days)
    
    for op in operations:
        if op['type'] == 'fixed': # เวลา Setup คงที่
            times = [op['time']] * len(days)
        elif op['type'] == 'variable': # เวลาต่อชิ้น
            times = [u * op['time_per_unit'] for u in por_units]
        
        crp_data[op['name']] = times
        total_time = [t + curr for t, curr in zip(total_time, times)]
        
    crp_data['Total time required (min)'] = total_time
    crp_data['Capacity (min)'] = [capacity_per_day] * len(days)
    
    df = pd.DataFrame(crp_data).T
    df.columns = df.iloc[0].astype(int)
    display(HTML("<b>การประเมินความต้องการกำลังการผลิต (CRP)</b>"))
    display(df[1:])
    
    overloads = [d for d, req in zip(days, total_time) if req > capacity_per_day]
    if overloads:
        print(f"⚠️ แจ้งเตือน: กำลังการผลิตไม่พอในวันที่ {overloads}")

# 6.1

In [7]:
# ---------------------------------------------------------
# 3. กำหนดข้อมูลตัวอย่างที่ 6.1 (กรอกข้อมูลตรงนี้)
# ---------------------------------------------------------
periods = [1, 2, 3, 4, 5, 6]

# แผนการผลิตหลักของผลิตภัณฑ์ A
mps = {
    'A': [0, 0, 0, 100, 0, 100]
}

# โครงสร้างผลิตภัณฑ์
bom = {
    'A': {'C': 3, 'D': 2}
}

# ข้อมูล Item Master (ตารางที่ 6.2)
item_master = {
    'A': {'OH': 10,  'SR': [0, 0, 0, 0, 0, 0],   'LotSize': 'L4L', 'LT': 3},
    'C': {'OH': 300, 'SR': [0, 0, 0, 0, 0, 0],   'LotSize': 150,   'LT': 2},
    'D': {'OH': 200, 'SR': [0, 250, 0, 0, 0, 0], 'LotSize': 250,   'LT': 3} 
}

# ลำดับการรัน Parent -> Child
process_order = ['A', 'C', 'D']

# ---------------------------------------------------------
# 4. เรียกใช้การทำงาน
# ---------------------------------------------------------
run_mrp_system_slide_logic(periods, mps, bom, item_master, process_order)

ช่วงเวลา,1,2,3,4,5,6
Gross Requirement (GR),0,0,0,100,0,100
Scheduled Receipts (SR),0,0,0,0,0,0
On Hand (OH),10,10,10,10,0,0
Net Requirement (NR),-10,-10,-10,90,0,100
Planned Order Receipts uncorrected,0,0,0,90,0,100
"Planned Order Receipts (POR), corrected for lot size",0,0,0,90,0,100
Planned End Inventory (PEI),10,10,10,0,0,0
Planned Order Releases (PREL),90,0,100,0,0,0


ช่วงเวลา,1,2,3,4,5,6
Gross Requirement (GR),0,270,0,300,0,0
Scheduled Receipts (SR),0,0,0,0,0,0
On Hand (OH),300,300,30,30,30,30
Net Requirement (NR),-300,-30,-30,270,-30,-30
Planned Order Receipts uncorrected,0,0,0,270,0,0
"Planned Order Receipts (POR), corrected for lot size",0,0,0,300,0,0
Planned End Inventory (PEI),300,30,30,30,30,30
Planned Order Releases (PREL),0,300,0,0,0,0


ช่วงเวลา,1,2,3,4,5,6
Gross Requirement (GR),180,0,200,0,0,0
Scheduled Receipts (SR),0,250,0,0,0,0
On Hand (OH),200,20,270,70,70,70
Net Requirement (NR),-20,-270,-70,-70,-70,-70
Planned Order Receipts uncorrected,0,0,0,0,0,0
"Planned Order Receipts (POR), corrected for lot size",0,0,0,0,0,0
Planned End Inventory (PEI),20,270,70,70,70,70
Planned Order Releases (PREL),0,0,0,0,0,0


# 6.2

In [5]:
# ---------------------------------------------------------
# 3. กำหนดข้อมูลตัวอย่างที่ 6.1 (กรอกข้อมูลตรงนี้)
# ---------------------------------------------------------
periods = [1, 2, 3, 4, 5, 6]

# แผนการผลิตหลักของผลิตภัณฑ์ A
mps = {
    'A': [0, 0, 0, 100, 0, 100],
    'B': [0, 0, 0, 0, 0, 200]
}

# โครงสร้างผลิตภัณฑ์
bom = {
    'A': {'C': 3, 'D': 2},
    'B': { 'D': 3}
}

# ข้อมูล Item Master (ตารางที่ 6.2)
item_master = {
    'A': {'OH': 10,  'SR': [0, 0, 0, 0, 0, 0],   'LotSize': 'L4L', 'LT': 3},
    'B': {'OH': 5,  'SR': [0, 0, 0, 0, 0, 0],   'LotSize': 'L4L', 'LT': 2},
    'C': {'OH': 140, 'SR': [0, 0, 0, 0, 0, 0],   'LotSize': 150,   'LT': 2},
    'D': {'OH': 200, 'SR': [0, 250, 0, 0, 0, 0], 'LotSize': 250,   'LT': 3} 
}

# ลำดับการรัน Parent -> Child
process_order = ['A', 'B', 'C', 'D']

# ---------------------------------------------------------
# 4. เรียกใช้การทำงาน
# ---------------------------------------------------------
run_mrp_system_slide_logic(periods, mps, bom, item_master, process_order)

ช่วงเวลา,1,2,3,4,5,6
Gross Requirement (GR),0,0,0,100,0,100
Scheduled Receipts (SR),0,0,0,0,0,0
On Hand (OH),10,10,10,10,0,0
Net Requirement (NR),-10,-10,-10,90,0,100
Planned Order Receipts uncorrected,0,0,0,90,0,100
"Planned Order Receipts (POR), corrected for lot size",0,0,0,90,0,100
Planned End Inventory (PEI),10,10,10,0,0,0
Planned Order Releases (PREL),90,0,100,0,0,0


ช่วงเวลา,1,2,3,4,5,6
Gross Requirement (GR),0,0,0,0,0,200
Scheduled Receipts (SR),0,0,0,0,0,0
On Hand (OH),5,5,5,5,5,5
Net Requirement (NR),-5,-5,-5,-5,-5,195
Planned Order Receipts uncorrected,0,0,0,0,0,195
"Planned Order Receipts (POR), corrected for lot size",0,0,0,0,0,195
Planned End Inventory (PEI),5,5,5,5,5,0
Planned Order Releases (PREL),0,0,0,195,0,0


ช่วงเวลา,1,2,3,4,5,6
Gross Requirement (GR),0,270,0,300,0,0
Scheduled Receipts (SR),0,0,0,0,0,0
On Hand (OH),140,140,20,20,20,20
Net Requirement (NR),-140,130,-20,280,-20,-20
Planned Order Receipts uncorrected,0,130,0,280,0,0
"Planned Order Receipts (POR), corrected for lot size",0,150,0,300,0,0
Planned End Inventory (PEI),140,20,20,20,20,20
Planned Order Releases (PREL),0,300,0,0,0,0


ช่วงเวลา,1,2,3,4,5,6
Gross Requirement (GR),180,0,785,0,0,0
Scheduled Receipts (SR),0,250,0,0,0,0
On Hand (OH),200,20,270,235,235,235
Net Requirement (NR),-20,-270,515,-235,-235,-235
Planned Order Receipts uncorrected,0,0,515,0,0,0
"Planned Order Receipts (POR), corrected for lot size",0,0,750,0,0,0
Planned End Inventory (PEI),20,270,235,235,235,235
Planned Order Releases (PREL),0,0,0,0,0,0


In [8]:
days = [1, 2, 3, 4, 5]
por_units = [120, 150, 120, 120, 130]
capacity_per_day = 714 # นาที (ได้จาก 2 กะ * 7 ชม. * 60 นาที * 0.85)

# กำหนดขั้นตอนการผลิต (Operations) เพิ่มหรือลบ list ตรงนี้ได้เลย
operations = [
    {'name': 'Set Up Time 1 (Min)', 'type': 'fixed', 'time': 10},
    {'name': 'Stamping/Bending (3.5 min/unit)', 'type': 'variable', 'time_per_unit': 3.5},
    {'name': 'Set Up Time 2 (Min)', 'type': 'fixed', 'time': 15},
    {'name': 'Painting (2.5 min/unit)', 'type': 'variable', 'time_per_unit': 2.5}
]

# รันระบบ CRP
run_crp_system(days, por_units, operations, capacity_per_day)

Schedule (day),1,2,3,4,5
POR (Unit),120.0,150.0,120.0,120.0,130.0
Set Up Time 1 (Min),10.0,10.0,10.0,10.0,10.0
Stamping/Bending (3.5 min/unit),420.0,525.0,420.0,420.0,455.0
Set Up Time 2 (Min),15.0,15.0,15.0,15.0,15.0
Painting (2.5 min/unit),300.0,375.0,300.0,300.0,325.0
Total time required (min),745.0,925.0,745.0,745.0,805.0
Capacity (min),714.0,714.0,714.0,714.0,714.0


⚠️ แจ้งเตือน: กำลังการผลิตไม่พอในวันที่ [1, 2, 3, 4, 5]
